# Phase 4: Deep Learning Models

Self-contained: everything is inline, no hidden deps. Smoke test on 50 states, full training on remote.

In [ ]:
import sys, os, io, pickle, tarfile, urllib.request, json, urllib.parse
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datetime import datetime

DATA = Path('dataset-task2')
CACHE = Path('.cache')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

## 1. Load Data

In [ ]:
articles = pd.read_csv(DATA / 'articles.csv')
categories = pd.read_csv(DATA / 'categories.csv')
train = pd.read_csv(DATA / 'states_train.csv')
test = pd.read_csv(DATA / 'states_test.csv')
print(f'Articles: {len(articles)}, Train: {len(train)}, Test: {len(test)}')

## 2. Load / Build Adjacency

Tries cache, then auto-downloads from SNAP if missing.

In [ ]:
def load_adjacency():
    # Try cache first, then auto-download
    for p in [CACHE / 'wikispeedia_adj.pkl']:
        if p.exists():
            with open(p, 'rb') as f:
                return pickle.load(f)
    
    # Auto-download from SNAP
    print('Downloading Wikispeedia graph from SNAP...')
    url = 'https://snap.stanford.edu/data/wikispeedia/wikispeedia_paths-and-graph.tar.gz'
    resp = urllib.request.urlopen(url)
    with tarfile.open(fileobj=io.BytesIO(resp.read()), mode='r:gz') as tar:
        f = tar.extractfile('wikispeedia_paths-and-graph/links.tsv')
        content = f.read()
    links = pd.read_csv(io.BytesIO(content), sep='\t', skiprows=14,
                        header=None, names=['source', 'target'])
    
    title_to_id = dict(zip(articles['title'].str.strip(), articles['article_id']))
    links['source_id'] = links['source'].apply(
        lambda x: urllib.parse.unquote(x).replace('_', ' ').strip()).map(title_to_id)
    links['target_id'] = links['target'].apply(
        lambda x: urllib.parse.unquote(x).replace('_', ' ').strip()).map(title_to_id)
    links = links.dropna(subset=['source_id', 'target_id'])
    
    adj = {i: [] for i in range(len(articles))}
    for _, r in links.iterrows():
        adj[int(r['source_id'])].append(int(r['target_id']))
    adj = {k: list(set(v)) for k, v in adj.items()}
    with open(CACHE / 'wikispeedia_adj.pkl', 'wb') as f:
        pickle.dump(adj, f)
    return adj

adj = load_adjacency()
n_edges = sum(len(v) for v in adj.values())
print(f'Adjacency: {len(adj)} nodes, {n_edges} edges')

## 3. Compute / Load Article Embeddings

In [ ]:
if (CACHE / 'article_embeddings.npy').exists():
    embeddings = np.load(CACHE / 'article_embeddings.npy')
else:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = model.encode(
        articles['title'].tolist(), show_progress_bar=True, batch_size=64)
    np.save(CACHE / 'article_embeddings.npy', embeddings)

print(f'Embeddings: {embeddings.shape}')

## 4. Category Encoding

In [ ]:
all_cats = sorted(categories['category'].unique())
cat_to_idx = {c: i for i, c in enumerate(all_cats)}
cat_enc = np.zeros((len(articles), len(all_cats)), dtype=np.float32)
for _, r in categories.iterrows():
    cat_enc[r['article_id'], cat_to_idx[r['category']]] = 1.0
n_cats = cat_enc.shape[1]
print(f'Categories: {n_cats}')

## 5. Candidate Dataset

In [ ]:
class CandidateDataset(Dataset):
    def __init__(self, states_df, adj, embeddings, cat_enc):
        self.samples = []
        for _, r in states_df.iterrows():
            curr, tgt, nxt = r['current_article_id'], r['target_article_id'], r['next_article_id']
            links = adj.get(curr, [])
            if not links:
                continue
            cemb = torch.tensor(embeddings[curr], dtype=torch.float)
            temb = torch.tensor(embeddings[tgt], dtype=torch.float)
            ccats = torch.tensor(cat_enc[curr], dtype=torch.float)
            tcats = torch.tensor(cat_enc[tgt], dtype=torch.float)
            for link in links:
                self.samples.append({
                    'curr_emb': cemb, 'tgt_emb': temb,
                    'cand_emb': torch.tensor(embeddings[link], dtype=torch.float),
                    'curr_cats': ccats, 'tgt_cats': tcats,
                    'cand_cats': torch.tensor(cat_enc[link], dtype=torch.float),
                    'out_deg': float(len(adj.get(link, []))),
                    'n_links': float(len(links)),
                    'label': 1.0 if link == nxt else 0.0,
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        s = self.samples[i]
        return (s['curr_emb'], s['tgt_emb'], s['cand_emb'],
                s['curr_cats'], s['tgt_cats'], s['cand_cats'],
                torch.tensor(s['out_deg']), torch.tensor(s['n_links']),
                torch.tensor(s['label']))

## 6. MLP Scorer

In [ ]:
class MLPScorer(nn.Module):
    def __init__(self, emb_dim=384, n_cats=0, hidden=128):
        super().__init__()
        d = emb_dim * 3 + n_cats * 3 + 2  # 3 embeddings + 3 catvecs + 2 graph feats
        self.net = nn.Sequential(
            nn.Linear(d, hidden), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, cemb, temb, pemb, ccats, tcats, pcats, out_deg, n_links):
        x = torch.cat([
            cemb, temb, pemb, ccats, tcats, pcats,
            out_deg.unsqueeze(-1).float(), n_links.unsqueeze(-1).float(),
        ], dim=-1)
        return self.net(x).squeeze(-1)

## 7. GCN Scorer

2-layer GCN + MLP scorer, no external deps (just torch).

In [ ]:
class GCNConv(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.w = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, x, adj_norm):
        return F.relu(adj_norm @ self.w(x))


class GCNScorer(nn.Module):
    def __init__(self, node_dim=384, hidden=128, out_dim=64, n_cats=0):
        super().__init__()
        self.conv1 = GCNConv(node_dim + n_cats, hidden)
        self.conv2 = GCNConv(hidden, out_dim)
        d = out_dim * 3 + 2
        self.mlp = nn.Sequential(
            nn.Linear(d, hidden), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, cnode, tnode, pnode, out_deg, n_links, adj_norm, full_feats):
        h = self.conv1(full_feats, adj_norm)
        h = self.conv2(h, adj_norm)
        cemb = h[cnode]
        temb = h[tnode]
        pemb = h[pnode]
        x = torch.cat([
            cemb, temb, pemb,
            out_deg.unsqueeze(-1).float(), n_links.unsqueeze(-1).float(),
        ], dim=-1)
        return self.mlp(x).squeeze(-1)

## 8. Build Normalized Adjacency

In [ ]:
def build_norm_adj(adj, n_nodes, device='cpu'):
    m = np.zeros((n_nodes, n_nodes), dtype=np.float32)
    for s, ts in adj.items():
        for t in ts:
            m[s, t] = 1.0
    deg = m.sum(1, keepdims=True)
    deg[deg == 0] = 1
    d = np.power(deg, -0.5).flatten()
    return torch.tensor(m * d[np.newaxis, :] * d[:, np.newaxis], device=device)

n_nodes = len(articles)
adj_norm = build_norm_adj(adj, n_nodes, device) if device == 'cuda' else None
print(f'Adj norm built' if adj_norm is not None else 'Skipped adj norm (cpu — use GCN on GPU only)')

## 9. Training Loop

In [ ]:
def train_epoch(model, loader, optimizer, loss_fn, device, is_gnn=False, adj_norm=None, full_feats=None):
    model.train()
    total = 0
    for batch in loader:
        curr_emb, tgt_emb, cand_emb, curr_cats, tgt_cats, cand_cats, out_deg, n_links, labels = [
            x.to(device) for x in batch]
        if is_gnn:
            cnode = torch.zeros(curr_emb.size(0), dtype=torch.long, device=device)
            tnode = torch.zeros(curr_emb.size(0), dtype=torch.long, device=device)
            pnode = torch.zeros(curr_emb.size(0), dtype=torch.long, device=device)
            scores = model(cnode, tnode, pnode, out_deg, n_links, adj_norm, full_feats)
        else:
            scores = model(curr_emb, tgt_emb, cand_emb, curr_cats, tgt_cats, cand_cats, out_deg, n_links)
        loss = loss_fn(scores, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item()
    return total / len(loader)

## 10. Prediction

In [ ]:
@torch.no_grad()
def predict(model, states_df, adj, embeddings, cat_enc, device='cpu', is_gnn=False, adj_norm=None, full_feats=None):
    model.eval()
    preds = []
    for _, r in tqdm(states_df.iterrows(), total=len(states_df), desc='Predict'):
        curr, tgt = r['current_article_id'], r['target_article_id']
        links = adj.get(curr, [])
        if not links:
            preds.append(tgt)
            continue
        B = len(links)
        cemb = torch.tensor(embeddings[curr], dtype=torch.float, device=device).unsqueeze(0).expand(B, -1)
        temb = torch.tensor(embeddings[tgt], dtype=torch.float, device=device).unsqueeze(0).expand(B, -1)
        pemb = torch.tensor(embeddings[links], dtype=torch.float, device=device)
        ccats = torch.tensor(cat_enc[curr], dtype=torch.float, device=device).unsqueeze(0).expand(B, -1)
        tcats = torch.tensor(cat_enc[tgt], dtype=torch.float, device=device).unsqueeze(0).expand(B, -1)
        pcats = torch.tensor(cat_enc[links], dtype=torch.float, device=device)
        deg = torch.tensor([len(adj.get(l, [])) for l in links], dtype=torch.float, device=device)
        n_links = torch.tensor(float(B), dtype=torch.float, device=device).expand(B)
        if is_gnn:
            cnode = torch.full((B,), curr, dtype=torch.long, device=device)
            tnode = torch.full((B,), tgt, dtype=torch.long, device=device)
            pnode = torch.tensor(links, dtype=torch.long, device=device)
            scores = model(cnode, tnode, pnode, deg, n_links, adj_norm, full_feats)
        else:
            scores = model(cemb, temb, pemb, ccats, tcats, pcats, deg, n_links)
        preds.append(links[scores.argmax().item()])
    return np.array(preds)

## 11. Smoke Test — MLPScorer

50 states, 1 epoch, verify forward/backward/predict.

In [ ]:
tiny = train.sample(50, random_state=42)
ds = CandidateDataset(tiny, adj, embeddings, cat_enc)
loader = DataLoader(ds, batch_size=16, shuffle=True)

mlp = MLPScorer(emb_dim=384, n_cats=n_cats, hidden=128).to(device)
opt = torch.optim.Adam(mlp.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()

loss = train_epoch(mlp, loader, opt, loss_fn, device, is_gnn=False)
print(f'MLPScorer — 1 epoch loss: {loss:.4f} | params: {sum(p.numel() for p in mlp.parameters()):,}')

preds = predict(mlp, test.head(5), adj, embeddings, cat_enc, device)
print(f'MLPScorer predictions on 5 test states: {preds}')

## 12. Smoke Test — GCNScorer (GPU only)

GCN needs dense adjacency matrix (4604x4604) — GPU memory required.

In [ ]:
if device == 'cuda':
    full_feats = torch.cat([
        torch.tensor(embeddings, dtype=torch.float, device=device),
        torch.tensor(cat_enc, dtype=torch.float, device=device),
    ], dim=-1)

    gcn = GCNScorer(node_dim=384, hidden=128, out_dim=64, n_cats=n_cats).to(device)
    opt = torch.optim.Adam(gcn.parameters(), lr=1e-3)
    loss = train_epoch(gcn, loader, opt, loss_fn, device, is_gnn=True, adj_norm=adj_norm, full_feats=full_feats)
    print(f'GCNScorer — 1 epoch loss: {loss:.4f} | params: {sum(p.numel() for p in gcn.parameters()):,}')

    preds = predict(gcn, test.head(5), adj, embeddings, cat_enc, device,
                    is_gnn=True, adj_norm=adj_norm, full_feats=full_feats)
    print(f'GCNScorer predictions on 5 test states: {preds}')
else:
    print('Skipping GCN — needs GPU for dense adj matrix')

## 13. Submission Helper

In [ ]:
def make_submission(state_ids, preds, path):
    pd.DataFrame({'state_id': state_ids, 'predicted_next_article_id': preds}).to_csv(path, index=False)

def validate(path):
    expected = pd.read_csv(DATA / 'sample_submission.csv')
    actual = pd.read_csv(path)
    assert list(actual.columns) == list(expected.columns)
    assert len(actual) == len(expected)
    assert list(actual['state_id']) == list(expected['state_id'])
    print(f'Validated: {path}')

## 14. Full Training — MLPScorer (run on remote server)

In [ ]:
# UNCOMMENT to run full training:

# ds = CandidateDataset(train, adj, embeddings, cat_enc)
# loader = DataLoader(ds, batch_size=256, shuffle=True, num_workers=4)

# mlp = MLPScorer(emb_dim=384, n_cats=n_cats, hidden=256).to(device)
# opt = torch.optim.AdamW(mlp.parameters(), lr=1e-3, weight_decay=1e-5)
# loss_fn = nn.BCEWithLogitsLoss()
# sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=5)

# for epoch in range(5):
#     loss = train_epoch(mlp, loader, opt, loss_fn, device)
#     sched.step()
#     print(f'Epoch {epoch}: loss={loss:.4f}')

# preds = predict(mlp, test, adj, embeddings, cat_enc, device)
# sub_dir = Path('submissions') / f'{datetime.now().strftime("%Y%m%d_%H%M%S")}_dl_mlp'
# sub_dir.mkdir(parents=True, exist_ok=True)
# make_submission(test['state_id'].values, preds, sub_dir / 'submission.csv')
# validate(sub_dir / 'submission.csv')
# torch.save(mlp.state_dict(), sub_dir / 'model.pt')
# print(f'Saved: {sub_dir / "submission.csv"}')